In [1]:
import sys
import torch
from ultralytics import YOLO

# ---------------------------------------------------------
# [설정] 사용자 환경에 맞게 경로 확인
# ---------------------------------------------------------
sys.path.append('..') 
from models.swin_yolo26 import SwinYOLO26  # Swin-YOLO26 모듈

IMG_PATH = '../data/samples/204_16_5e636120-4c7e-48e2-aae8-e3592973947d.jpg'
YOLO_WEIGHTS = '../runs/baseline_yolo11m_640_한글라벨밀림/weights/best.pt'
SWIN_WEIGHTS = '../runs/exp_model_swin_yolo26m-3/weights/best.pt'

# 1. 경로 설정 확인
print("DEBUG: 설정 시작")
sys.path.append('..') 
from models.swin_yolo26 import SwinYOLO26

# 파라미터 수 계산 함수
def count_parameters(model):
    # 학습 가능한 파라미터만 계산 (requires_grad=True)
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# 2. Baseline YOLO 로드
print("DEBUG: YOLO(YOLO_WEIGHTS) 초기화 시작...")
# 만약 아래 줄에서 멈춘다면, ultralytics가 가중치를 로드하며 네트워크를 체크하는 중에 걸린 것입니다.
baseline_model = YOLO(YOLO_WEIGHTS) 
print("DEBUG: YOLO 로드 완료")

# 3. 제안 모델 초기화
print("DEBUG: SwinYOLO26 초기화 시작...")
proposed_model = SwinYOLO26(swin_size='n', yolo_size='m', num_classes=7)
print("DEBUG: SwinYOLO26 초기화 완료")

yolo_params = count_parameters(baseline_model.model)

# Proposed: Swin-YOLO26m
ckpt = torch.load(SWIN_WEIGHTS, map_location='cpu')
state_dict = ckpt.get('ema', ckpt.get('model', ckpt))
if hasattr(state_dict, 'state_dict'): 
    state_dict = state_dict.state_dict()
proposed_model.load_state_dict(state_dict, strict=False)
swin_params = count_parameters(proposed_model)

print(f"📊 Model Parameter Summary")
print(f"--------------------------------------------------")
print(f"YOLO11m (Baseline)    : {yolo_params / 1e6:.2f} M")
print(f"Swin-YOLO26m (Ours)   : {swin_params / 1e6:.2f} M")
print(f"--------------------------------------------------")

def print_model_summary(model, name):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[{name}] Summary")
    print(f"  - Total Parameters:     {total_params / 1e6:.2f} M")
    print(f"  - Trainable Parameters: {trainable_params / 1e6:.2f} M")
    print("-" * 40)

# 실행
print_model_summary(baseline_model.model, "YOLO11m")
print_model_summary(proposed_model, "Swin-YOLO26m")

c:\Users\trust\anaconda3\envs\swin-yolo\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEBUG: 설정 시작
DEBUG: YOLO(YOLO_WEIGHTS) 초기화 시작...
DEBUG: YOLO 로드 완료
DEBUG: SwinYOLO26 초기화 시작...
Overriding model.yaml nc=80 with nc=7

                   from  n    params  module                                       arguments                     
  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  2                  -1  1    111872  ultralytics.nn.modules.block.C3k2            [128, 256, 1, True, 0.25]     
  3                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  4                  -1  1    444928  ultralytics.nn.modules.block.C3k2            [256, 512, 1, True, 0.25]     
  5                  -1  1   2360320  ultralytics.nn.modules.conv.Conv             [512, 512, 3, 2]              
  6                  -1  1   1380352  ultralytics.nn.modules.block.C

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_32172\3505666641.py:39: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(SWIN_WEIGHTS, map_location='c

📊 Model Parameter Summary
--------------------------------------------------
YOLO11m (Baseline)    : 0.00 M
Swin-YOLO26m (Ours)   : 49.99 M
--------------------------------------------------
[YOLO11m] Summary
  - Total Parameters:     20.06 M
  - Trainable Parameters: 0.00 M
----------------------------------------
[Swin-YOLO26m] Summary
  - Total Parameters:     49.99 M
  - Trainable Parameters: 49.99 M
----------------------------------------


In [2]:
import torch
from thop import profile
from models.swin_yolo26 import SwinYOLO26
from ultralytics import YOLO

# 1. 모델 준비
baseline_model = YOLO('../runs/baseline_yolo11m_640_한글라벨밀림/weights/best.pt')
proposed_model = SwinYOLO26(swin_size='n', yolo_size='m', num_classes=7)

# 2. FLOPs 및 파라미터 계산을 위한 더미 입력 (640x640 고정)
dummy_input = torch.randn(1, 3, 640, 640)

# 3. 계산 함수
def get_model_stats(model, input_size, name):
    # 파라미터 계산 (Total)
    total_params = sum(p.numel() for p in model.parameters())
    
    # FLOPs 계산 (thop 활용)
    # 모델을 잠시 eval 모드로 두고 연산량 측정
    model.eval()
    flops, params = profile(model, inputs=(input_size,), verbose=False)
    
    print(f"[{name}]")
    print(f"  - Total Parameters: {total_params / 1e6:.2f} M")
    print(f"  - GFLOPs (at 640x640): {flops / 1e9:.2f} G")
    print("-" * 30)

# 실행 (YOLO 모델은 ultralytics 내부 구조상 .model 접근)
get_model_stats(baseline_model.model, dummy_input, "YOLO11m")
get_model_stats(proposed_model, dummy_input, "Swin-YOLO26m")

Overriding model.yaml nc=80 with nc=7

                   from  n    params  module                                       arguments                     
  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  2                  -1  1    111872  ultralytics.nn.modules.block.C3k2            [128, 256, 1, True, 0.25]     
  3                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  4                  -1  1    444928  ultralytics.nn.modules.block.C3k2            [256, 512, 1, True, 0.25]     
  5                  -1  1   2360320  ultralytics.nn.modules.conv.Conv             [512, 512, 3, 2]              
  6                  -1  1   1380352  ultralytics.nn.modules.block.C3k2            [512, 512, 1, True]           
  7                  -1  1   2360320  ultralytics

In [4]:
# [수정] 실제 사용 중인 레이어의 파라미터만 합산하는 코드
def get_active_params(model):
    # 1. 사용 중인 레이어 목록 (Swin Backbone + Bridge + Head)
    # YOLO Neck/Head는 10번 인덱스 이후부터 사용함
    active_params = 0
    
    # Swin Backbone
    active_params += sum(p.numel() for p in model.swin_backbone.parameters() if p.requires_grad)
    # Bridge
    active_params += sum(p.numel() for p in model.bridge.parameters() if p.requires_grad)
    # YOLO Neck/Head (10번 인덱스 이후만)
    for i in range(10, len(model.model)):
        active_params += sum(p.numel() for p in model.model[i].parameters() if p.requires_grad)
        
    return active_params

# 실행
active_params = get_active_params(proposed_model)
print(f"📊 Actual Active Parameters: {active_params / 1e6:.2f} M")

📊 Actual Active Parameters: 40.63 M
